# Track B — Bring your own machine

**No labels needed. No test set. Nothing to beat.**

Track A asks whether a model transfers to a bearing it has never seen. Track B asks a
different and more useful question: can you say anything at all about a machine *you* have
access to, using no fault labels?

**What this is:** low-frequency exploratory anomaly detection using a phone accelerometer.

**What this is not:** a reproduction of the Paderborn bearing pipeline. Paderborn sampled at
64 kHz and band-passed 1–10 kHz to catch the resonances that bearing impacts excite. A phone
at 200–400 Hz has a Nyquist limit of 100–200 Hz, so those resonances are not merely
attenuated — they are physically unmeasurable. You are looking at low-frequency vibration:
shaft harmonics, imbalance, looseness, structural response. That can still be informative,
and finding out how informative is the point.

You need a phone and a rotating machine — a water pump, a maize mill, a generator, a fan, an
air conditioner. Any accelerometer logging app will do.

## 1. Record — safely

**Safety first, and this is not negotiable.**

- Never modify, loosen, unbalance or otherwise interfere with a machine to create a fault.
  Deliberately introducing a defect into rotating equipment is dangerous and can injure
  people and destroy the machine.
- Do not attach anything to, or place any part of yourself near, a rotating shaft, belt,
  coupling, fan blade or any other moving part.
- Only record machines you are authorised to be near. Ask the owner or operator first.
- If there is a guard, it stays on. Tape the phone to a stationary housing on the outside.

Two recordings, one minute each:

- **normal** — the machine running as it usually does
- **a different condition** — obtained only through variation that is already safe and
  permitted:
  - a different speed or load setting the machine already offers
  - startup or run-down versus steady state
  - before versus after a scheduled service
  - a different but similar machine, including one already known to be worn
  - the same machine at a different time of day, or under different ambient conditions

Tape the phone to the bearing housing rather than the frame, where you can do so safely.
Note the shaft speed if you can — a tachometer app, or the motor nameplate.

Export as CSV. If you cannot record anything today, `demo/` contains a **simulated** signal
so the code still runs. It is not a measurement and no conclusion may be drawn from it.

In [ ]:
import numpy as np
from byo_machine import prepare, anomaly_scores

# --- load your recording ------------------------------------------------------
# expects a single column of acceleration, or a CSV with a column you name below
FS = 400          # <-- your sampling rate in Hz, from the app
normal = np.loadtxt("demo/normal.csv", delimiter=",")
other  = np.loadtxt("demo/other.csv",  delimiter=",")

print(f"normal {len(normal)/FS:.1f} s   other {len(other)/FS:.1f} s")

## 2. Preprocess

Band-pass, Hilbert envelope, then resample to 512 samples per 256 ms window — the same
*shape* of pipeline as the challenge data, operating on a completely different frequency
range.

`prepare()` defaults its band to roughly 15–95% of your Nyquist frequency. At 400 Hz that is
about 30–190 Hz. The challenge data used 1–10 kHz. Envelope demodulation still makes sense
here — it will pick up amplitude modulation of whatever low-frequency content exists — but do
not describe your result as bearing-resonance analysis, because it is not.

In [ ]:
W_normal = prepare(normal, fs=FS)
W_other  = prepare(other,  fs=FS)
print(f"{W_normal.shape[0]} normal windows, {W_other.shape[0]} other windows, "
      f"{W_normal.shape[1]} samples each")

## 3. Score without labels

Fit a Gaussian to normal operation, then measure how far other windows sit from it by
Mahalanobis distance. No fault labels anywhere. This is the only starting point available for
almost every machine in the world, which is the point of the track.

**One thing to get right.** The threshold is the 95th percentile of the distances of the
windows used to fit the reference. If you then score those same windows, about 5% will be
flagged *by definition* — that is arithmetic, not validation. So split your normal recording:
fit on one half, and estimate the false-alarm rate on the other half, which the reference has
never seen.

**A second thing to get right.** Each window is 512 numbers, and a one-minute recording gives
only a few hundred windows — fewer samples than dimensions. A covariance estimated that way
does not generalise, and the held-out false-alarm rate blows up. `anomaly_scores` takes an
`n_components` argument to reduce the dimension first; the cell below sweeps it so you can
see the trade-off rather than being handed a number.

Even with both fixed, this is a rough estimate from one short recording under one set of
conditions. It is not an independent validation of a deployable detector, and you should say
so. The `demo/` figures come from a *simulated* signal and mean nothing about any real
machine.

In [ ]:
# Split the NORMAL recording: fit the reference on one half, estimate false alarms
# on the other. Scoring the windows you fitted on builds the 5% in by construction.
half = len(W_normal) // 2
W_fit, W_heldout = W_normal[:half], W_normal[half:]
print(f"{len(W_fit)} windows to fit, {len(W_heldout)} held out, {W_fit.shape[1]} dimensions each\n")

# A window is 512 numbers and you have a few hundred of them, so the covariance must
# be reduced first. How far to reduce is a choice - and it is YOUR choice to justify.
print(f"{'n_components':>13}{'held-out normal':>18}{'other condition':>18}")
print(f"{'':13}{'(false alarms)':>18}{'(flagged)':>18}")
for k in [8, 16, 32, 64, None]:
    held, thr = anomaly_scores(W_fit, W_heldout, n_components=k)
    other, _  = anomaly_scores(W_fit, W_other,   n_components=k)
    print(f"{str(k):>13}{100*(held>thr).mean():>17.1f}%{100*(other>thr).mean():>17.1f}%")

print("\nNo free lunch: reduce hard and you stop raising false alarms but also stop")
print("detecting anything; reduce little and everything is flagged, including normal")
print("data the reference has never seen. Pick a value, say why, and report both columns.")

## 4. The part that is actually judged

Answer these four in your talk. They are worth more than any number above.

1. **What machine, and what changed between the two recordings?** Be specific. "A borehole
   pump at 2900 rpm, second recording with the impeller partly blocked" is a finding.
   "Some vibration data" is not.
2. **What is your false-alarm rate on held-out normal data, and how many windows was it
   estimated from?** If it is 40%, the method is not deployable on that machine yet, and
   saying so is a stronger result than hiding it. If you only had enough data to fit and not
   to hold out, say that instead of reporting the in-sample 5%.
3. **Is the separation real, or something trivial?** A different load, a different mounting,
   a phone that moved between recordings, a different ambient temperature — all of these
   separate beautifully and mean nothing. Which of them can you rule out, and how?
   Remember also that at 200–400 Hz you are not measuring bearing resonances at all, so a
   bearing-fault interpretation needs more evidence than a distance score.
4. **Who owns this recording, and where should it live?** If a model is going to be built
   for African machines, this is where that data starts.

If your answer to 3 is "I cannot rule it out", say that. It is the correct answer more often
than not, and it is the difference between a demonstration and a claim.